# 从零实现 CLIP 风格双编码器：图文对比学习与检索

本 Notebook 不导入 OpenCLIP、Transformers 或 torchvision 模型，而是手写：

- 小型 CNN image encoder；
- token embedding、位置编码和 manual self-attention text encoder；
- 双投影头、L2 normalize、可学习 temperature；
- 对称 InfoNCE、图找文/文找图指标和可信检索制品。

合成图形只验证架构和损失。高 Recall 不能代表真实开放词表理解、OCR、文化偏差、版权或安全能力。

## 1. 离线运行与数据边界

CPU 单线程、固定种子。train/validation/test 图像噪声独立；模型只按 validation 观察，不用 test 选 checkpoint。真实多模态数据必须按来源/原图 family 切分，避免同图裁剪泄漏。

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="The pynvml package is deprecated", category=FutureWarning)

from copy import deepcopy
from hashlib import sha256
import json
import math
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED=4201
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.set_num_threads(1)
DEVICE=torch.device("cpu")
assert DEVICE.type=="cpu"
print({"torch":torch.__version__,"seed":SEED})

## 2. 合成图文 pair 与 tokenizer

六类 16x16 灰度图分别包含竖线、横线、主/副对角线、十字和方框；文本是 `[BOS] 形状 token [EOS]`。每个 split 使用不同噪声。预处理均值/标准差只从 train 图像计算。

一批中每个 pair ID 唯一，适合标准 CLIP diagonal InfoNCE。若一个语义有多张图或多条 caption，其他正例会被当 false negative，应改 multi-positive loss 或去重采样。

In [ ]:
TOKENS={"[PAD]":0,"[BOS]":1,"[EOS]":2,"竖线":3,"横线":4,"主对角":5,"副对角":6,"十字":7,"方框":8}
CLASS_WORDS=["竖线","横线","主对角","副对角","十字","方框"]

def pattern_image(label):
    image=torch.zeros(16,16)
    if label==0: image[:,7:9]=1
    elif label==1: image[7:9,:]=1
    elif label==2:
        for i in range(16): image[i,max(0,i-1):min(16,i+2)]=1
    elif label==3:
        for i in range(16):
            j=15-i; image[i,max(0,j-1):min(16,j+2)]=1
    elif label==4: image[:,7:9]=1; image[7:9,:]=1
    elif label==5: image[2:14,2:4]=1; image[2:14,12:14]=1; image[2:4,2:14]=1; image[12:14,2:14]=1
    else: raise ValueError("unknown_label")
    return image

def make_split(seed, copies):
    generator=torch.Generator().manual_seed(seed)
    images=[]; labels=[]
    for copy in range(copies):
        for label in range(6):
            base=pattern_image(label)
            images.append((base+0.06*torch.randn(base.shape,generator=generator)).clamp(0,1))
            labels.append(label)
    images=torch.stack(images)[:,None]
    texts=torch.tensor([[TOKENS["[BOS]"],TOKENS[CLASS_WORDS[y]],TOKENS["[EOS]"],TOKENS["[PAD]"]] for y in labels])
    lengths=torch.full((len(labels),),3,dtype=torch.long)
    return images, texts, lengths, torch.tensor(labels)

train_images_raw,train_texts,train_text_lengths,train_labels=make_split(SEED,1)
valid_images_raw,valid_texts,valid_text_lengths,valid_labels=make_split(SEED+1,2)
test_images_raw,test_texts,test_text_lengths,test_labels=make_split(SEED+2,2)
image_mean=train_images_raw.mean(); image_std=train_images_raw.std()
train_images=(train_images_raw-image_mean)/image_std
valid_images=(valid_images_raw-image_mean)/image_std
test_images=(test_images_raw-image_mean)/image_std
assert train_images.shape==(6,1,16,16)
assert valid_images.shape==test_images.shape==(12,1,16,16)
assert train_texts[:,3].eq(TOKENS["[PAD]"]).all()
assert image_std>0
print({"train_pairs":len(train_images),"validation":len(valid_images),"test":len(test_images)})

## 3. 图像 CNN 与手写文本 self-attention

image encoder 用卷积与 global average pooling；text encoder 手写 QKV attention，key padding 在 softmax 前屏蔽，并从 EOS 位置取句向量。两个 encoder 输出维度可以不同，随后各自投影到共享空间。

文本是双向编码，不使用 causal mask。改变 padding token 内容但保持 length 不变，不应改变 EOS 表示。

In [ ]:
def valid_mask42(lengths,width):
    if lengths.ndim!=1 or bool((lengths<=0).any()) or bool((lengths>width).any()): raise ValueError("invalid_text_lengths")
    return torch.arange(width)[None,:] < lengths[:,None]

class ImageEncoder42(nn.Module):
    def __init__(self,out_dim=32):
        super().__init__(); self.out_dim=out_dim
        self.features=nn.Sequential(nn.Conv2d(1,8,3,padding=1),nn.SiLU(),nn.MaxPool2d(2),
                                    nn.Conv2d(8,16,3,padding=1),nn.SiLU(),nn.MaxPool2d(2),
                                    nn.Conv2d(16,out_dim,3,padding=1),nn.SiLU())
    def forward(self,images):
        if images.ndim!=4 or images.shape[1:]!=(1,16,16) or not torch.isfinite(images).all(): raise ValueError("image_contract")
        return self.features(images).mean(dim=(-2,-1))

class TextSelfAttention42(nn.Module):
    def __init__(self,dim,heads):
        super().__init__()
        if dim%heads: raise ValueError("head_divisibility")
        self.dim,self.heads,self.head_dim=dim,heads,dim//heads; self.scale=self.head_dim**-0.5
        self.qkv=nn.Linear(dim,3*dim); self.output=nn.Linear(dim,dim)
    def forward(self,x,mask):
        if x.ndim!=3 or x.shape[-1]!=self.dim or mask.shape!=x.shape[:2] or mask.dtype!=torch.bool:
            raise ValueError("attention_shape_or_mask_contract")
        if not bool(mask.any(dim=1).all()):
            raise ValueError("attention_requires_valid_key")
        batch,steps,_=x.shape
        qkv=self.qkv(x).view(batch,steps,3,self.heads,self.head_dim)
        q,k,v=qkv.unbind(2); q,k,v=(z.transpose(1,2) for z in (q,k,v))
        score=(q@k.transpose(-2,-1))*self.scale
        weight=torch.softmax(score.masked_fill(~mask[:,None,None,:],float("-inf")),dim=-1)
        out=(weight@v).transpose(1,2).reshape(batch,steps,self.dim)
        return self.output(out)*mask.unsqueeze(-1),weight

class TextEncoder42(nn.Module):
    def __init__(self,vocab_size,dim=24,heads=3,max_length=8):
        super().__init__(); self.vocab_size,self.dim,self.max_length=vocab_size,dim,max_length
        self.token=nn.Embedding(vocab_size,dim,padding_idx=0); self.position=nn.Embedding(max_length,dim)
        self.norm1=nn.LayerNorm(dim); self.attention=TextSelfAttention42(dim,heads)
        self.norm2=nn.LayerNorm(dim); self.ffn=nn.Sequential(nn.Linear(dim,2*dim),nn.GELU(),nn.Linear(2*dim,dim))
        self.final=nn.LayerNorm(dim)
    def forward(self,tokens,lengths):
        if tokens.ndim!=2 or tokens.shape[1]>self.max_length or lengths.shape!=(tokens.shape[0],): raise ValueError("text_contract")
        if bool((tokens<0).any()) or bool((tokens>=self.vocab_size).any()): raise ValueError("token_out_of_range")
        mask=valid_mask42(lengths,tokens.shape[1]).to(tokens.device)
        positions=torch.arange(tokens.shape[1],device=tokens.device)[None,:]
        x=(self.token(tokens)+self.position(positions))*mask.unsqueeze(-1)
        attended,weights=self.attention(self.norm1(x),mask); x=(x+attended)*mask.unsqueeze(-1)
        x=(x+self.ffn(self.norm2(x)))*mask.unsqueeze(-1); x=self.final(x)*mask.unsqueeze(-1)
        eos_index=lengths-1
        pooled=x[torch.arange(len(tokens)),eos_index]
        return pooled,weights

image_probe42=ImageEncoder42(); text_probe42=TextEncoder42(len(TOKENS))
assert image_probe42(train_images).shape==(6,32)
pooled42,attention42=text_probe42(train_texts,train_text_lengths)
assert pooled42.shape==(6,24) and attention42.shape==(6,3,4,4)
altered_texts42=train_texts.clone(); altered_texts42[:,3]=TOKENS["方框"]
assert torch.allclose(text_probe42(train_texts,train_text_lengths)[0],text_probe42(altered_texts42,train_text_lengths)[0],atol=1e-6)

attention_oracle42=TextSelfAttention42(4,2)
with torch.no_grad():
    attention_oracle42.qkv.weight.copy_(torch.cat([torch.eye(4)]*3)); attention_oracle42.qkv.bias.zero_()
    attention_oracle42.output.weight.copy_(torch.eye(4)); attention_oracle42.output.bias.zero_()
oracle_x42=torch.tensor([[[1.,0.,1.,0.],[0.,1.,0.,2.]]]); oracle_mask42=torch.tensor([[True,True]])
oracle_out42,oracle_weight42=attention_oracle42(oracle_x42,oracle_mask42)
oracle_q42=oracle_x42.view(1,2,2,2).transpose(1,2)
expected_weight42=torch.softmax((oracle_q42@oracle_q42.transpose(-2,-1))/math.sqrt(2),dim=-1)
assert torch.allclose(oracle_weight42,expected_weight42,atol=1e-6)
assert not torch.allclose(expected_weight42,torch.softmax(oracle_q42@oracle_q42.transpose(-2,-1),dim=-1))
assert torch.allclose(oracle_out42,(expected_weight42@oracle_q42).transpose(1,2).reshape(1,2,4),atol=1e-6)
try:
    attention_oracle42(oracle_x42,torch.zeros_like(oracle_mask42))
    raise AssertionError("all-masked attention must fail")
except ValueError as error:
    assert str(error)=="attention_requires_valid_key"
assert bool(oracle_mask42.any())

## 4. CLIP 双塔、归一化与温度

投影后做 L2 normalize，故相似度是 cosine。`logit_scale` 以 log 参数保存，forward 中 clamp 后 exponentiate，避免无界温度导致 overflow。图找文 logits 为 `[B_image,B_text]`，反向检索就是转置。

下面用直接 dot-product oracle 检查尺度、归一化和矩阵方向。

In [ ]:
class CLIP42(nn.Module):
    def __init__(self,vocab_size,embed_dim=24):
        super().__init__(); self.vocab_size,self.embed_dim=vocab_size,embed_dim
        self.image_feature_dim,self.text_feature_dim,self.text_heads=32,24,3
        self.image_encoder=ImageEncoder42(self.image_feature_dim)
        self.text_encoder=TextEncoder42(vocab_size,self.text_feature_dim,self.text_heads)
        self.image_projection=nn.Linear(self.image_feature_dim,embed_dim,bias=False)
        self.text_projection=nn.Linear(self.text_feature_dim,embed_dim,bias=False)
        self.logit_scale=nn.Parameter(torch.tensor(math.log(1/0.07)))
    def encode_image(self,images): return F.normalize(self.image_projection(self.image_encoder(images)),dim=-1,eps=1e-8)
    def encode_text(self,tokens,lengths): return F.normalize(self.text_projection(self.text_encoder(tokens,lengths)[0]),dim=-1,eps=1e-8)
    def forward(self,images,tokens,lengths):
        image_embeddings=self.encode_image(images); text_embeddings=self.encode_text(tokens,lengths)
        scale=self.logit_scale.clamp(math.log(1/100),math.log(100)).exp()
        return scale*(image_embeddings@text_embeddings.T),image_embeddings,text_embeddings,scale

model42=CLIP42(len(TOKENS))
probe_logits42,probe_image42,probe_text42,probe_scale42=model42(train_images,train_texts,train_text_lengths)
assert probe_logits42.shape==(6,6)
assert torch.allclose(probe_image42.norm(dim=1),torch.ones(6),atol=1e-6)
assert torch.allclose(probe_text42.norm(dim=1),torch.ones(6),atol=1e-6)
assert torch.allclose(probe_logits42,probe_scale42*(probe_image42@probe_text42.T),atol=1e-6)
assert 0.01<=float(probe_scale42)<=100

## 5. 对称 InfoNCE 与 false negative 边界

标准 CLIP batch 是一一对应 pair。令单位归一化后的图像/文本向量为 `u_i,v_j`，温度倒数为 `s=exp(logit_scale)`，相似度矩阵 `L_ij=s*u_i^T v_j`；对称目标是 `0.5 * (CE(L, arange(B)) + CE(L^T, arange(B)))`。同样重排图文 pair 不应改变 loss；只重排一侧则会破坏 diagonal label。

若 batch 含重复语义正例，diagonal loss 会把它们视作负例。真实训练需用 pair/group ID 构造 multi-positive objective，不能靠增大 batch 掩盖标注问题。

In [ ]:
def symmetric_clip_loss(logits):
    if logits.ndim!=2 or logits.shape[0]!=logits.shape[1] or logits.shape[0]<2: raise ValueError("paired_square_logits_required")
    labels=torch.arange(logits.shape[0],device=logits.device)
    return 0.5*(F.cross_entropy(logits,labels)+F.cross_entropy(logits.T,labels))

identity_logits=torch.eye(3)*5
bad_logits=-identity_logits
assert symmetric_clip_loss(identity_logits)<symmetric_clip_loss(bad_logits)
permutation=torch.tensor([2,0,1])
assert torch.allclose(symmetric_clip_loss(identity_logits),symmetric_clip_loss(identity_logits[permutation][:,permutation]))
asymmetric_logits42=torch.tensor([[2.0,-1.0,0.5],[0.2,1.1,-0.7],[-0.4,2.3,0.1]])
asymmetric_labels42=torch.arange(3)
row_ce42=F.cross_entropy(asymmetric_logits42,asymmetric_labels42)
column_ce42=F.cross_entropy(asymmetric_logits42.T,asymmetric_labels42)
assert not torch.allclose(row_ce42,column_ce42)
assert torch.allclose(symmetric_clip_loss(asymmetric_logits42),0.5*(row_ce42+column_ce42))

def multi_positive_clip_loss(logits, group_ids):
    if logits.shape!=(len(group_ids),len(group_ids)):
        raise ValueError("multi_positive_shape")
    positives=group_ids[:,None].eq(group_ids[None,:])
    def direction(value,mask):
        log_probability=F.log_softmax(value,dim=1)
        return -torch.logsumexp(log_probability.masked_fill(~mask,float("-inf")),dim=1).mean()
    return 0.5*(direction(logits,positives)+direction(logits.T,positives.T))

duplicate_logits42=torch.tensor([[6.,6.,0.],[6.,6.,0.],[0.,0.,6.]])
duplicate_groups42=torch.tensor([0,0,1])
assert multi_positive_clip_loss(duplicate_logits42,duplicate_groups42)<symmetric_clip_loss(duplicate_logits42)
try:
    symmetric_clip_loss(torch.zeros(2,3)); raise AssertionError("rectangular paired loss must fail")
except ValueError:
    pass
probe_loss42=symmetric_clip_loss(probe_logits42); probe_loss42.backward()
assert model42.image_projection.weight.grad.abs().sum()>0
assert model42.text_projection.weight.grad.abs().sum()>0
assert model42.logit_scale.grad.abs()>0

## 6. 受控对比训练

每个 train batch 恰好六个唯一 pair。固定训练步数，不用 test 调参；validation 只观察类级 retrieval。记录参数变化、loss 下降和温度有限性。真实 CLIP 需要巨大且去重的数据、分布式 all-gather、混合精度和 false-negative 策略。

In [ ]:
torch.manual_seed(SEED+3)
model42=CLIP42(len(TOKENS),embed_dim=24)
optimizer42=torch.optim.AdamW(model42.parameters(),lr=0.01,weight_decay=1e-4)
initial42={k:v.detach().clone() for k,v in model42.state_dict().items()}; losses42=[]
for step in range(61):
    model42.train(); optimizer42.zero_grad(set_to_none=True)
    logits,_,_,scale=model42(train_images,train_texts,train_text_lengths)
    loss=symmetric_clip_loss(logits); loss.backward()
    if step==0: first_grad42=torch.sqrt(sum((p.grad**2).sum() for p in model42.parameters() if p.grad is not None))
    torch.nn.utils.clip_grad_norm_(model42.parameters(),5.0); optimizer42.step(); losses42.append(float(loss.detach()))
model42.eval()
changed42=any(not torch.equal(initial42[k],v) for k,v in model42.state_dict().items())
assert changed42 and first_grad42>0
assert losses42[-1]<0.15*losses42[0]
assert math.isfinite(float(model42.logit_scale))
print({"loss":[round(losses42[0],4),round(losses42[-1],4)],"temperature":round(float(model42.logit_scale.exp().reciprocal()),4)})

## 7. 双向全候选 retrieval

test 图像对六条 canonical 文本做全候选排序，文本则对全部 test 图像检索。相关性按 class，而不是只承认某个 diagonal instance。报告 Recall@1 与 MRR；评估前不使用 test label 选参数。

这同时说明训练 diagonal pair ID 与评估 semantic relevance 可能不同，指标的 gold 定义必须明确。

In [ ]:
def relevance_metrics42(matrix,query_labels,candidate_labels):
    reciprocal=[]; hit=[]
    for row,label in zip(matrix,query_labels.tolist()):
        order=torch.argsort(row,descending=True).tolist()
        ranks=[rank for rank,index in enumerate(order,1) if int(candidate_labels[index])==label]
        if not ranks: raise ValueError("query_without_relevant_candidate")
        reciprocal.append(1/ranks[0]); hit.append(ranks[0]==1)
    return float(np.mean(hit)),float(np.mean(reciprocal))

@torch.no_grad()
def retrieval_metrics42(model,images,image_labels,texts,text_lengths,text_labels):
    model.eval(); image_emb=model.encode_image(images); text_emb=model.encode_text(texts,text_lengths); scores=image_emb@text_emb.T
    return (relevance_metrics42(scores,image_labels,text_labels),
            relevance_metrics42(scores.T,text_labels,image_labels),scores)

asymmetric_scores42=torch.tensor([[3.,0.],[0.,3.],[0.,2.]])
asym_image_labels42=torch.tensor([0,1,0]); asym_text_labels42=torch.tensor([0,1])
asym_i2t42=relevance_metrics42(asymmetric_scores42,asym_image_labels42,asym_text_labels42)
asym_t2i42=relevance_metrics42(asymmetric_scores42.T,asym_text_labels42,asym_image_labels42)
assert math.isclose(asym_i2t42[0],2/3) and math.isclose(asym_i2t42[1],5/6)
assert asym_t2i42==(1.0,1.0)

canonical_texts=train_texts; canonical_lengths=train_text_lengths; canonical_labels=train_labels
i2t42,t2i42,test_scores42=retrieval_metrics42(model42,test_images,test_labels,canonical_texts,canonical_lengths,canonical_labels)
valid_i2t42,valid_t2i42,_=retrieval_metrics42(model42,valid_images,valid_labels,canonical_texts,canonical_lengths,canonical_labels)
print({"validation_i2t":valid_i2t42,"test_i2t":i2t42,"test_t2i":t2i42})
assert i2t42[0]>=0.8 and i2t42[1]>=0.85
assert t2i42[0]>=0.8 and t2i42[1]>=0.85
assert test_scores42.shape==(12,6)

## 8. 可信图文检索制品

模型权重必须和 tokenizer、canonical 文本库、图像预处理统计、训练图像快照以及 config 一起发布。公开搜索接口只接 model version，从内部 registry 加载并逐项校验；篡改权重或候选文本都 fail closed。

服务还需要租户 ACL、内容安全、候选版本、批量上限和 trace。embedding 不是匿名数据，不能跳过访问控制。

In [ ]:
def hash_tensor42(value):
    array=value.detach().cpu().contiguous().numpy(); return sha256(str(array.dtype).encode()+json.dumps(list(array.shape)).encode()+array.tobytes()).hexdigest()
def hash_model42(model):
    digest=sha256()
    for name,value in sorted(model.state_dict().items()): digest.update(name.encode()); digest.update(hash_tensor42(value).encode())
    return digest.hexdigest()
def candidate_records_hash42(texts,lengths,labels,candidate_ids):
    digest=sha256()
    for value in (texts,lengths,labels): digest.update(hash_tensor42(value).encode())
    digest.update(json.dumps(list(candidate_ids),ensure_ascii=False,separators=(",",":")).encode())
    return digest.hexdigest()
def config_model42(model):
    return {"vocab_size":model.vocab_size,"embed_dim":model.embed_dim,
            "image_feature_dim":model.image_feature_dim,"text_feature_dim":model.text_feature_dim,
            "text_heads":model.text_heads}
def bundle_hash42(value): return sha256(json.dumps({k:v for k,v in value.items() if k!="bundle_sha256"},sort_keys=True,ensure_ascii=False).encode()).hexdigest()

artifact42={"model_version":"clip-shapes-v1","architecture":"CLIP42","config":config_model42(model42),
            "state_sha256":hash_model42(model42),"tokenizer":TOKENS,
            "code_version":"clip-teaching-v1","candidate_ids":CLASS_WORDS,
            "label_map":{str(i):name for i,name in enumerate(CLASS_WORDS)},
            "canonical_text_sha256":hash_tensor42(canonical_texts),
            "canonical_length_sha256":hash_tensor42(canonical_lengths),
            "canonical_label_sha256":hash_tensor42(canonical_labels),
            "candidate_records_sha256":candidate_records_hash42(canonical_texts,canonical_lengths,canonical_labels,CLASS_WORDS),
            "train_image_sha256":hash_tensor42(train_images_raw),
            "preprocess":{"mean":float(image_mean),"std":float(image_std),"size":[16,16],
                          "recipe":"train-global-zscore-v1"}}
artifact42["bundle_sha256"]=bundle_hash42(artifact42)
_REGISTRY42={artifact42["model_version"]:{"model":model42,"artifact":deepcopy(artifact42),
                                          "texts":canonical_texts.clone(),"lengths":canonical_lengths.clone(),
                                          "labels":canonical_labels.clone(),
                                          "train_images":train_images_raw.clone()}}

def load_clip42(version):
    if version not in _REGISTRY42: raise KeyError("unknown_model_version")
    entry=_REGISTRY42[version]; model,artifact=entry["model"],entry["artifact"]
    if type(model) is not CLIP42 or artifact["architecture"]!=type(model).__name__: raise TypeError("architecture_mismatch")
    if artifact["bundle_sha256"]!=bundle_hash42(artifact): raise RuntimeError("bundle_mismatch")
    if artifact["config"]!=config_model42(model) or artifact["state_sha256"]!=hash_model42(model): raise RuntimeError("model_mismatch")
    if artifact["canonical_text_sha256"]!=hash_tensor42(entry["texts"]): raise RuntimeError("candidate_snapshot_mismatch")
    if artifact["canonical_length_sha256"]!=hash_tensor42(entry["lengths"]): raise RuntimeError("candidate_snapshot_mismatch")
    if artifact["canonical_label_sha256"]!=hash_tensor42(entry["labels"]): raise RuntimeError("candidate_snapshot_mismatch")
    if len(artifact["candidate_ids"])!=len(entry["texts"]): raise RuntimeError("candidate_id_mismatch")
    if artifact["candidate_records_sha256"]!=candidate_records_hash42(entry["texts"],entry["lengths"],entry["labels"],artifact["candidate_ids"]):
        raise RuntimeError("candidate_records_mismatch")
    inverse_tokenizer={int(token_id):token for token,token_id in artifact["tokenizer"].items()}
    for row,length,label,candidate_id in zip(entry["texts"],entry["lengths"],entry["labels"],artifact["candidate_ids"]):
        length=int(length); label=int(label)
        if length!=3 or int(row[0])!=artifact["tokenizer"]["[BOS]"] or int(row[length-1])!=artifact["tokenizer"]["[EOS]"]:
            raise RuntimeError("candidate_text_contract")
        if inverse_tokenizer.get(int(row[1]))!=candidate_id or artifact["label_map"].get(str(label))!=candidate_id:
            raise RuntimeError("candidate_semantic_mismatch")
    if artifact["train_image_sha256"]!=hash_tensor42(entry["train_images"]): raise RuntimeError("train_snapshot_mismatch")
    expected_mean=float(entry["train_images"].mean()); expected_std=float(entry["train_images"].std())
    preprocess=artifact.get("preprocess",{})
    if (preprocess.get("recipe")!="train-global-zscore-v1" or preprocess.get("size")!=[16,16]
            or not math.isclose(preprocess.get("mean",float("nan")),expected_mean,rel_tol=0,abs_tol=1e-8)
            or not math.isclose(preprocess.get("std",float("nan")),expected_std,rel_tol=0,abs_tol=1e-8)
            or expected_std<=0):
        raise RuntimeError("preprocess_contract_mismatch")
    return entry

@torch.no_grad()
def search_text42(raw_images,top_k=2,version="clip-shapes-v1"):
    entry=load_clip42(version); artifact=entry["artifact"]; model=entry["model"]
    raw=torch.as_tensor(raw_images,dtype=torch.float32)
    if raw.ndim!=4 or raw.shape[1:]!=(1,16,16) or not 1<=raw.shape[0]<=16 or not 1<=top_k<=6 or not torch.isfinite(raw).all(): raise ValueError("search_request_contract")
    normalized=(raw-artifact["preprocess"]["mean"])/artifact["preprocess"]["std"]
    scores=model.encode_image(normalized)@model.encode_text(entry["texts"],entry["lengths"]).T
    indices=torch.topk(scores,top_k,dim=1).indices
    words=[[artifact["candidate_ids"][int(i)] for i in row] for row in indices]
    return words,{"model_version":version,"bundle_sha256":artifact["bundle_sha256"],"candidate_count":len(entry["texts"])}

served42,trace42=search_text42(test_images_raw[:2],1)
assert served42[0][0]==CLASS_WORDS[int(test_labels[0])]
assert trace42["candidate_count"]==6
param42=next(model42.parameters()); backup42=param42.detach().clone()
try:
    with torch.no_grad(): param42.add_(0.2)
    try: search_text42(test_images_raw[:1]); raise AssertionError("tampered weights must fail")
    except RuntimeError as error: assert str(error)=="model_mismatch"
finally:
    with torch.no_grad(): param42.copy_(backup42)
candidate_backup42=_REGISTRY42["clip-shapes-v1"]["texts"][0,1].clone()
try:
    _REGISTRY42["clip-shapes-v1"]["texts"][0,1]=TOKENS["方框"]
    try: search_text42(test_images_raw[:1]); raise AssertionError("tampered candidates must fail")
    except RuntimeError as error: assert str(error)=="candidate_snapshot_mismatch"
finally:
    _REGISTRY42["clip-shapes-v1"]["texts"][0,1]=candidate_backup42
artifact_backup42=deepcopy(_REGISTRY42["clip-shapes-v1"]["artifact"])
try:
    changed42=_REGISTRY42["clip-shapes-v1"]["artifact"]
    changed42["candidate_ids"]=list(reversed(changed42["candidate_ids"]))
    changed42["candidate_records_sha256"]=candidate_records_hash42(
        _REGISTRY42["clip-shapes-v1"]["texts"],_REGISTRY42["clip-shapes-v1"]["lengths"],
        _REGISTRY42["clip-shapes-v1"]["labels"],changed42["candidate_ids"])
    changed42["bundle_sha256"]=bundle_hash42(changed42)
    try: search_text42(test_images_raw[:1]); raise AssertionError("misbound candidate ids must fail")
    except RuntimeError as error: assert str(error)=="candidate_semantic_mismatch"
finally:
    _REGISTRY42["clip-shapes-v1"]["artifact"]=deepcopy(artifact_backup42)
try:
    changed42=_REGISTRY42["clip-shapes-v1"]["artifact"]
    changed42["preprocess"]["mean"]+=1.0
    changed42["bundle_sha256"]=bundle_hash42(changed42)
    try: search_text42(test_images_raw[:1]); raise AssertionError("misderived preprocess must fail")
    except RuntimeError as error: assert str(error)=="preprocess_contract_mismatch"
finally:
    _REGISTRY42["clip-shapes-v1"]["artifact"]=artifact_backup42
try: search_text42(torch.full((1,1,16,16),float("nan"))); raise AssertionError("nonfinite image must fail")
except ValueError as error: assert str(error)=="search_request_contract"
assert trace42["candidate_count"] == len(CLASS_WORDS)

## 9. 面试总结与来源

CLIP 的关键链路是独立图文编码、共享空间归一化、temperature-scaled 相似度和双向对比目标。必须同时讲清 batch 内负例假设、重复正例、全候选评估、预处理版本和索引/权限边界。

- Radford et al., *Learning Transferable Visual Models From Natural Language Supervision*：https://arxiv.org/abs/2103.00020
- van den Oord et al., *Representation Learning with Contrastive Predictive Coding*：https://arxiv.org/abs/1807.03748
- PyTorch `normalize` 文档：https://pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html

本例没有大规模分布式训练、真实 tokenizer、zero-shot prompt ensemble、ANN 索引或安全审核。

In [ ]:
assert type(model42).__name__=="CLIP42"
assert type(model42.image_encoder).__name__=="ImageEncoder42"
assert type(model42.text_encoder.attention).__name__=="TextSelfAttention42"
assert losses42[-1]<losses42[0]
assert i2t42[0]>=0.8 and t2i42[0]>=0.8
assert artifact42["tokenizer"]==TOKENS
assert bundle_hash42(artifact42)==artifact42["bundle_sha256"]
assert hash_model42(model42)==artifact42["state_sha256"]
assert trace42["model_version"]=="clip-shapes-v1"
print("CLIP 双编码器、InfoNCE、双向检索与可信制品回归全部通过。")